# Exercises 01–06

**Prof. Daniel de Abreu Pereira Uhr**

**Course:** Statistics and Econometrics with Python



## The exporter — Brazilian exporter

The company tracks its share of regional exports (%), volume (thousands of 60-kg bags), and exchange rate (BRL per USD), monthly from 2016 to 2025. Data are simulated and mechanisms are visible in the generator. Use laboratory knowledge to evaluate the procedure, while distinguishing it from what an analyst could infer from an unknown path. Both editions retain Brazil and BRL/USD.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.tsa.stattools import acf

rng = np.random.default_rng(2026)
aquecimento, T = 300, 120
months = np.arange(T)


def ar1(persistencia, desvio):
    choques = rng.normal(0, desvio, size=aquecimento + T)
    u = np.zeros(aquecimento + T)
    for t in range(1, aquecimento + T):
        u[t] = persistencia * u[t - 1] + choques[t]
    return u[aquecimento:]


exportadora = pd.DataFrame({
    "mes": pd.period_range("2016-01", periods=T, freq="M"),
    "participacao": np.round(20 + ar1(0.6, 1.0), 2),
    "volume": np.round(np.exp(np.log(50) + 0.005 * months + ar1(0.5, 0.05)), 2),
    "cambio": np.round(np.exp(np.log(3.4) + np.cumsum(np.r_[0, rng.normal(0.0015, 0.025, size=T - 1)])), 3),
})

## Exercise 01 — The graphs

Plot all three series with units and dates. Describe levels, trends, and persistent movements. Which conclusion is descriptive and which requires knowing the generator?

In [ ]:
# Your work here

## Exercise 02 — The two halves

Compare means and standard deviations of share, log volume, and log exchange rate across the two halves. Why do similar means not establish stationarity?

In [ ]:
# Your work here

## Exercise 03 — The correlograms

Compute ACF through 12 months and highlight 1, 6, and 12. Does the correlogram distinguish a deterministic trend from a unit root? Explain the plotted bands.

In [ ]:
# Your work here

## Exercise 04 — Removing the trend

Fit linear trends to log volume and log exchange rate. Compare residuals and ACF at 1, 6, and 12. Identify what 100 times a residual means.

In [ ]:
# Your work here

## Exercise 05 — Shocks and half-lives

Fit AR(1) models to share and log-volume deviations. Compute half-lives and responses at 1, 3, and 6 months to −2 percentage points in share and −10% in volume. Compare with −10% in the exchange rate.

In [ ]:
# Your work here

## Exercise 06 — Challenge: dispersion across paths

Simulate the prescribed 1,000 paths. Compare dispersion at months 12, 60, and 119 and final exchange-rate percentiles 5/95. Distinguish the band from a confidence interval and a conditional forecast interval.

In [ ]:
# Your work here

# Exercises 07–13

**Prof. Daniel de Abreu Pereira Uhr**

**Course:** Statistics and Econometrics with Python



## The time-series coffee roaster — Coffee roaster

The roaster records monthly revenue (USD thousands) and the price of a 60-kg bag of green coffee (USD), from 2016 to 2025. Data are simulated. The task is to measure growth and develop purchasing scenarios; the data contain no margin, purchased quantity, or inventory policy. Do not invent these values.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.tsa.stattools import acf, adfuller

rng = np.random.default_rng(2026)
aquecimento, T = 300, 120
months = np.arange(T)

choques = rng.normal(0, 0.04, size=aquecimento + T)
u = np.zeros(aquecimento + T)
for t in range(1, aquecimento + T):
    u[t] = 0.6 * u[t - 1] + choques[t]
u = u[aquecimento:]

torrefacao = pd.DataFrame({
    "mes": pd.period_range("2016-01", periods=T, freq="M"),
    "faturamento": np.round(np.exp(np.log(800) + 0.006 * months + u), 1),
    "preco_cafe": np.round(np.exp(np.log(900) + np.cumsum(np.r_[0, rng.normal(0.004, 0.05, size=T - 1)])), 1),
})

## Exercise 07 — Growth rates

Compare monthly log and exact growth and their largest difference. Compute log annualization, compounded annualization, and observed 12-month growth. Identify units and overlapping windows.

In [ ]:
# Your work here

## Exercise 08 — Deterministic terms

Examine the logs and justify c or ct before inspecting p-values. Which alternative is relevant? What limitations remain?

In [ ]:
# Your work here

## Exercise 09 — ADF in logs

Apply ADF with the proposed terms. Record maximum and selected lags, effective n, statistic, p, and critical values; interpret without accepting H0 because it is not rejected.

In [ ]:
# Your work here

## Exercise 10 — ADF in differences

Test log differences with an intercept. Record lost observations and explain why rejection in differences does not automatically choose the business transformation.

In [ ]:
# Your work here

## Exercise 11 — Differencing or detrending

Compare ACF at 1, 2, and 12 for detrended log revenue and its differences. Interpret dependence introduced by the transformation.

In [ ]:
# Your work here

## Exercise 12 — Shocks and the central path

Compute the remaining exact revenue decline after −20% at 1, 3, 6, and 12 months. For a 20% price increase, calculate the one-year central path using estimated drift. Why is it not automatically the arithmetic mean?

In [ ]:
# Your work here

## Exercise 13 — Challenge: specification and sample

Compare the three prespecified tests: c/120 months, ct/120, and ct/36. Interpret sensitivity and power without selecting a preferred test after seeing results.

In [ ]:
# Your work here

# Exercises 14–19

**Prof. Daniel de Abreu Pereira Uhr**

**Course:** Statistics and Econometrics with Python



## The economic consultancy — Macroeconomic scenarios

Use the historical macrodata CSV: United States, 1959Q1–2009Q3, 203 quarters. Real GDP is in billions of chained 2005 dollars, seasonally adjusted at an annual rate; annualized inflation and unemployment are percentages. Exclude only the initial inflation value as documented. These are real data used for teaching, not company records or a current forecast. Keep the dados folder when extracting the package.

In [ ]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.tsa.stattools import acf, adfuller

caminho = "data/ch16/macrodata.csv"
with open(caminho, "rb") as arquivo:
    print("File matches source note:", hashlib.sha256(arquivo.read()).hexdigest() == "7853fcc0d92a9b89b627cb6eb2d27faf4a00c7515e3b37a9a24001ef597ca8f0")

macro = pd.read_csv(caminho)
macro["trimestre"] = pd.PeriodIndex.from_fields(year=macro["year"], quarter=macro["quarter"], freq="Q")
macro = macro.set_index("trimestre")[["realgdp", "infl", "unemp"]]
log_pib = np.log(macro["realgdp"])
inflacao = macro["infl"].iloc[1:]
unemployment = macro["unemp"]
print(macro.head(3))
print(macro.tail(3))
assert len(macro)==203 and len(inflacao)==202
assert hashlib.sha256(Path(caminho).read_bytes()).hexdigest()=="7853fcc0d92a9b89b627cb6eb2d27faf4a00c7515e3b37a9a24001ef597ca8f0"

## Exercise 14 — GDP growth

Compute annualized quarterly log/exact growth, four-quarter log/exact growth, statistics, and dates of extremes. Explain why annual-rate GDP should not be multiplied by four again.

In [ ]:
# Your work here

## Exercise 15 — GDP trend

Fit the trend, calculate log/exact deviations and ACF at 1, 8, and 20. What future information entered the line? Is it correct to call the residual an output gap?

In [ ]:
# Your work here

## Exercise 16 — A unit root in GDP

Compare ADF for log GDP with ct and for growth with c. Write a conclusion that does not turn nonrejection into proof of a permanent loss.

In [ ]:
# Your work here

## Exercise 17 — Unemployment and inflation

Apply ADF with an intercept to both series. Explain initial inflation treatment, the limits of the bounded-rate argument, and the connection with Chapter 15.

In [ ]:
# Your work here

## Exercise 18 — Memo and business transfer

Write a memo for the economic consultancy: recovery scenarios and limits for hiring. Separate expansion, replacement, and existing vacancies; list required data and a review trigger. Do not invent a headcount.

In [ ]:
# Your work here

## Exercise 19 — Challenge: a spurious relationship

Reproduce the prespecified independent random walk and regressions in levels, with HAC, and in growth. Interpret R² and tests, distinguishing this experiment from a cointegration investigation.

In [ ]:
# Your work here